# DroneWaste Auto Training on Google Colab

Upload this notebook to Google Drive, open it with Colab, choose a GPU runtime, then run all cells. It will clone the DroneWaste fork, install dependencies, download the Zenodo dataset, and start training in order.

Dataset download is about 3.9 GB. The default run uses a short single-fold `yolov8n` training job so you can verify the runtime before launching a full cross-validation run.

In [ ]:
#@title 1. Configure the Colab run

REPO_URL = "https://github.com/Leo890728/dronewaste.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
PROJECT_DIR = "/content/dronewaste"  #@param {type:"string"}

ARCH = "yolov8"  #@param ["yolov8", "yolov12", "faster"]
MODEL = ""  #@param {type:"string"}
SITE_INDICES = "0"  #@param {type:"string"}
EPOCHS = 10  #@param {type:"integer"}
PATIENCE = 5  #@param {type:"integer"}
BATCH_SIZE = 16  #@param {type:"integer"}
IMG_SIZE = 640  #@param {type:"integer"}

USE_GOOGLE_DRIVE = False  #@param {type:"boolean"}
STORAGE = "/content"  #@param {type:"string"}
FORCE_RECLONE = False  #@param {type:"boolean"}
AUTO_INSTALL = True  #@param {type:"boolean"}
AUTO_DOWNLOAD = True  #@param {type:"boolean"}

WANDB_MODE = "offline"  #@param ["offline", "online", "disabled"]

print("Configuration ready.")

In [ ]:
#@title 2. Clone or update the project

from pathlib import Path
import os
import shutil
import subprocess

def run(command, cwd=None, env=None):
    location = f" in {cwd}" if cwd else ""
    print(f"$ {' '.join(command)}{location}")
    subprocess.run(command, cwd=cwd, env=env, check=True)

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

project_dir = Path(PROJECT_DIR)
if FORCE_RECLONE and project_dir.exists():
    shutil.rmtree(project_dir)

if not (project_dir / ".git").exists():
    if project_dir.exists():
        backup_dir = project_dir.with_name(project_dir.name + "_backup")
        print(f"{project_dir} exists but is not a git repo. Moving it to {backup_dir}.")
        if backup_dir.exists():
            shutil.rmtree(backup_dir)
        shutil.move(str(project_dir), str(backup_dir))
    run(["git", "clone", "--recursive", "--branch", BRANCH, REPO_URL, str(project_dir)])
else:
    run(["git", "fetch", "--all", "--prune"], cwd=project_dir)
    run(["git", "checkout", BRANCH], cwd=project_dir)
    run(["git", "pull", "--ff-only"], cwd=project_dir)
    run(["git", "submodule", "update", "--init", "--recursive"], cwd=project_dir)

os.chdir(project_dir)
print(f"Project is ready at {Path.cwd()}.")

In [ ]:
#@title 3. Install dependencies, download dataset, and train

from pathlib import Path
import os
import subprocess

project_dir = Path(PROJECT_DIR)
script_path = project_dir / "training" / "colab_train.sh"
if not script_path.exists():
    raise FileNotFoundError(f"Missing training script: {script_path}")

env = os.environ.copy()
env.update(
    {
        "ARCH": ARCH,
        "MODEL": MODEL,
        "SITE_INDICES": SITE_INDICES,
        "EPOCHS": str(EPOCHS),
        "PATIENCE": str(PATIENCE),
        "BATCH_SIZE": str(BATCH_SIZE),
        "IMG_SIZE": str(IMG_SIZE),
        "STORAGE": STORAGE,
        "AUTO_INSTALL": "1" if AUTO_INSTALL else "0",
        "AUTO_DOWNLOAD": "1" if AUTO_DOWNLOAD else "0",
        "WANDB_MODE": WANDB_MODE,
    }
)

print("Starting DroneWaste Colab pipeline...")
subprocess.run(["bash", str(script_path)], cwd=project_dir, env=env, check=True)

results_root = Path(STORAGE) / "kfold_results"
print(f"Done. Results are under: {results_root}")